# Test of doubly tapered Mestel disk simulation with EXP

$$
\Psi_o(R) = V_o^2 \log\frac{R}{R_o} \qquad \Sigma(R) = \frac{V_o^2}{2\pi G R} T_{inner}(R) T_{outer}(R)
$$
where
$$
T_{inner}(R) = \frac{R^\nu}{R_i^\nu | R^\nu}
$$
and
$$
T_{outer}(R) = \frac{R_o^\mu}{R_o^\mu+R^\mu}
$$

### pyEXP setup

In [ ]:
import os
import copy
import yaml
import time
import pyEXP
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker, cm, colors
from os.path import exists

plt.rcParams['figure.figsize'] = [12, 9]

# Change to the example's working directory
os.chdir('Zang')
print(os.getcwd())


In [ ]:
Ri = 1.0
Ro = 11.5
Vo = 1.0
Nu = 4.0
Mu = 5.0
Rmin = 0.001
Rmax = 50.0

def Tinner(R, nu):
    return np.power(R, nu)/(np.power(R, nu) + np.power(Ri, nu))

def Touter(R, mu):
    return np.power(Ro, mu)/(np.power(R, mu) + np.power(Ro, mu))

def Sigma(R, nu, mu):
    return Vo**2/(2*np.pi*R)*Tinner(R, nu)*Touter(R, mu)

R = np.arange(Rmin, Rmax, 0.01)
D = Sigma(R, Nu, Mu)
plt.loglog(R, D)
plt.xlabel('R')
plt.ylabel(r'$\Sigma(R)$')
plt.savefig('density.png')
plt.show()

## Save this profile to a file to make a the orthogonal functions for the field profiles

In [ ]:
file = open('density.model', 'w')
for index in range(len(R)):
    file.write('{:13.6e} {:13.6e}\n'.format(R[index], D[index]))
file.close()

## Make initial conditions

Initial conditions can be made with the `zangics` utility.  The options are:

In [ ]:
! zangics -h

Let's make a set of initial conditions for the disk illustrated above.

In [ ]:
N = 100000
S = 0.325
os.system('zangics --number {} --nu {} --mu {} --Rmin {} --Rmax {} --sigma {}'.format(N, Nu, Mu, Rmin, Rmax, S))

### Running the simulation

We have provided a sample `exp` configuration file in this working directory.  The default name for a configuration file is
`config.yml`.    

The basis build can take minutes.  If you want to experiment with the disk parameters, be prepared to wait a few minutes while 
the basis rebuilds.  The entire run will take under an hour; and under 10 minutes on a modern laptop.  So be prepared to 
take a short break. 

The `exp` code will run for 1000 steps.  You can estimate the total run time by looking at the last line of every step diagnostic 
stanza that estimates the time needed for each part of the caluculation.  The 'Total' field is the total run time in seconds per step.  E.g. if your computer is requiring 0.3 seconds per step, the completion time will be 10 minutes.

In [ ]:
%%time
! rm data/*
! mpirun exp config_vels.yml

### Configure the basis and read coefficients

In [ ]:
# Get the basis config
#
stream = open('config_vels.yml', 'r')
config = yaml.full_load(stream)
disk_config = yaml.dump(config['Components'][0]['force'])

# Get the runtag from the config file
#
runtag = config['Global']['runtag']

# Get the coefficient files
#
for v in config['Output']:
    if v['id'] == 'outcoef':
        coeffile = v['parameters']['filename']
    if v['id'] == 'outvel':
        vel_config = v
        
velcoef = 'velcoef.disk.' + config['Global']['runtag']
        
# Move to the data directory
#
os.chdir('data')

# Construct the basis instance
#
disk_basis = pyEXP.basis.Basis.factory(disk_config)

vel_config = """
id         : velocity
parameters :
  dof      : 2
  rmin     : 0.01
  rmax     : 50.0
  lmax     : 4
  nmax     : 12
  rmapping  : 1.0
  modelname : ../density.model
"""

vel_basis  = pyEXP.basis.Basis.factory(vel_config)

# Restore working directory
#
os.chdir('..')

### Read the coefficients

Both the simulation basis coefficients and the velocity coefficients are produced by the simulation we just run!

In [ ]:
coeffile1 = 'data/outcoef.disk.run0'
coefs0 = pyEXP.coefs.Coefs.factory(coeffile1, 1)
coefs  = coefs0.deepcopy()

vcoeffile = 'data/velcoef.disk.run0'
vcoefs0 = pyEXP.coefs.Coefs.factory(vcoeffile, 1)
vcoefs  = vcoefs0.deepcopy()

### Plot the coefficients for m=2

In [ ]:
data = coefs.getAllCoefs()
print(data.shape, len(coefs.Times()))
for i in range(data.shape[1]):
    plt.plot(coefs.Times(), np.real(data[2, i, :]), label=str(i)+"c")
    plt.plot(coefs.Times(), np.imag(data[2, i, :]), label=str(i)+"s")
plt.xlabel('Time')
plt.ylabel('Amplitude')
plt.legend()
plt.show()

## Plot the gravitational power

In [ ]:
pow = coefs.Power()
plt.rcParams.update({'font.size': 18})
plt.semilogy(coefs.Times(), pow[:,1], label='m=1')
plt.semilogy(coefs.Times(), pow[:,2], label='m=2')
plt.semilogy(coefs.Times(), pow[:,3], label='m=3')
plt.semilogy(coefs.Times(), pow[:,4], label='m=4')
plt.legend()
plt.xlabel('Time')
plt.ylabel('Power')
plt.title(r'Cold-disk instability for $\nu=2$, $\mu=2$, $R_i=1$, $R_o=20$, $\sigma=0.4$')
plt.show()

plt.plot(coefs.Times(), pow[:,1], label='m=1')
plt.plot(coefs.Times(), pow[:,2], label='m=2')
plt.plot(coefs.Times(), pow[:,3], label='m=3')
plt.plot(coefs.Times(), pow[:,4], label='m=4')
plt.legend()
plt.xlabel('Time')
plt.ylabel('Power')
plt.title(r'Cold-disk instability for $\nu=2$, $\mu=2$, $R_i=1$, $R_o=20$, $\sigma=0.4$')
plt.show()

In [ ]:
# Restricting non-zero coefficients to m=2
coefs = coefs0.deepcopy()

if True:
    for T in coefs.Times():
        data = coefs(T)
        for m in range(data.shape[0]):
            if m != 2: data[m, :] *= 0.0
        coefs.setMatrix(T, data)
else: # an alternative construction
    for T in coefs.Times():
        test = np.zeros(coefs(T).shape, dtype='complex128')
        test[2, :] = coefs(T)[2, :]
        coefs.setMatrix(T, test)

## Plot some density slices

We configure a 2d grid in x and y and generate the surfaces.

In [ ]:
times = coefs.Times()
rmax  = 10
ngrd  = 100
pmin  = [ -rmax, -rmax, 0.0]
pmax  = [  rmax,  rmax, 0.0]
grid  = [  ngrd,  ngrd,   0]

fields = pyEXP.field.FieldGenerator(times, pmin, pmax, grid)

print('Created fields instance')

surfaces = fields.slices(disk_basis, coefs)

### Get density min and max for level surface scaling

In [ ]:
dmin =  1e20
dmax = -1e20
for time in coefs.Times():
    dmin = min(dmin, np.min(surfaces[time]['dens']))
    dmax = max(dmax, np.max(surfaces[time]['dens']))
print(dmin, dmax)

In [ ]:
# Get the shape
keys = list(surfaces.keys())
nx = surfaces[keys[0]]['dens'].shape[0]
ny = surfaces[keys[0]]['dens'].shape[1]

# Make the mesh
x = np.linspace(-rmax, rmax, nx)
y = np.linspace(-rmax, rmax, ny)
xv, yv = np.meshgrid(x, y)

plt.rcParams.update({'font.size': 22})

### Render some density fields from the simulation-generated coefficients

In [ ]:
for index in range(0, len(keys)):
    fig, ax = plt.subplots(1, 1, figsize=(24, 20))
    key = keys[index]
    mat = surfaces[key]['dens']

    cont1 = ax.contour(xv, yv, mat.transpose(), colors='k')
    cont2 = ax.contourf(xv, yv, mat.transpose())
    plt.colorbar(cont2, ax=ax)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('T={:4.3f}'.format(key))
    plt.show()

## Orthogonality check for both the simulation and field bases

In [ ]:
ret = disk_basis.orthoCheck()
ctr = 0
for v in ret: 
    g = plt.imshow(np.log10(np.abs(v)))
    plt.colorbar(g)
    plt.title(str(ctr))
    plt.plot()
    plt.show()
    ctr += 1
    
ret = vel_basis.orthoCheck()
g = plt.imshow(np.log10(np.abs(ret)))
plt.colorbar(g)
plt.title('velocity basis')
plt.plot()
plt.show()

## Look at some basis functions

In [ ]:
logxmin = -1
logxmax =  np.log10(50) # this is edge of the Zang input profile
numgrid = 2000

disk_grid = disk_basis.getBasis(logxmin, logxmax, numgrid)
vel_grid  =  vel_basis.getBasis(logxmin, logxmax, numgrid)

In [ ]:
# Make a logarithmically space grid in radius
#
r = np.linspace(logxmin, logxmax, numgrid)
r = np.power(10.0, r)

for l in range(2,3):
    for n in range(6):
        plt.semilogx(r, disk_grid[l][n]['potential'], '-', label="n={}".format(n))
    plt.xlabel('r')
    plt.ylabel('potential')
    plt.title('m={}'.format(l))
    plt.legend()
    plt.show()
  
for n in range(10):
    plt.semilogx(r, vel_grid[n], '-', label="n={}".format(n))
plt.xlabel('r')
plt.ylabel('f')
plt.title('field basis')
plt.legend()
plt.show()

## Now, let's render some velocity fields

In [ ]:
# Restrict to m=2 coefficients

msel = 2
vcoefs = vcoefs0.deepcopy()
for T in vcoefs.Times():
    data = vcoefs(T)
    if False:
        for i in range(data.shape[0]):
            for m in range(data.shape[1]):
                if m != msel: data[i, m, :] *= 0
        vcoefs.setMatrix(T, data)
    else:
        test = np.zeros(data.shape, dtype='complex128')
        for i in range(data.shape[0]):
            test[i, msel, :] = data[i, msel, :]
        vcoefs.setMatrix(T, test)

In [ ]:
times = vcoefs.Times()
rmax  = 10
ngrd  = 100
pmin  = [ -rmax, -rmax, 0.0]
pmax  = [  rmax,  rmax, 0.0]
grid  = [  ngrd,  ngrd,   0]

fields = pyEXP.field.FieldGenerator(times, pmin, pmax, grid)

print('Created fields instance')

# Get surfaces for the restricted and full coefficient sets
vsurfaces  = fields.slices(vel_basis, vcoefs )
vsurfaces0 = fields.slices(vel_basis, vcoefs0)

In [ ]:
# Compute the min/max for the restricted reconstruction

vmin = {}
vmax = {}

for key in vsurfaces[times[0]]:
    vmin[key] = 1e20
    vmax[key] = -1e20

for time in coefs.Times():
    for k in vsurfaces[time]:
        vmin[k] = min(vmin[k], np.min(vsurfaces[time][k]))
        vmax[k] = max(vmax[k], np.max(vsurfaces[time][k]))
        
for key in vsurfaces0[times[0]]:
    print('{:20s} {:13.6e} {:13.6e}'.format(key, vmin[key], vmax[key]))

### Finally, display some field quantities: left is density, right is radial velocity

In [ ]:
key = 'v_R'

# Make the mesh
x = np.linspace(-rmax, rmax, nx)
y = np.linspace(-rmax, rmax, ny)
xv, yv = np.meshgrid(x, y)

for tim in vsurfaces:
    if tim in surfaces:
        fig, ax = plt.subplots(1, 2, figsize=(24, 12))
        mat = vsurfaces[tim]['v_R']/vsurfaces0[tim]['density'] # Divide <rho*v_R> by <rho>
        
        # Rotation curve as value 1.  Pick 15% of that to make a sane range for contour levels.
        v = np.linspace(-0.15, 0.15, 10)

        # The radial velocity
        cont1 = ax[1].contour(xv, yv, mat.transpose(), levels=v, colors='k')
        cont2 = ax[1].contourf(xv, yv, mat.transpose(), levels=v)
        plt.colorbar(cont2, ax=ax[1])
        ax[1].set_xlabel('x')
        ax[1].set_ylabel('y')
        ax[1].set_title('T={:4.3f}, ${:s}$'.format(tim, key))
    
        # The field density
        mat2 = vsurfaces[tim]['density']
        cont3 = ax[0].contour(xv, yv, mat2.transpose(), colors='k')
        cont4 = ax[0].contourf(xv, yv, mat2.transpose())
        plt.colorbar(cont4, ax=ax[0])
        ax[0].set_title(r'$\langle\Sigma\rangle$')
        plt.show()

The poorly determine center in the $v_R$ plots results from dividing $\langle\Sigma v_R\rangle$ by $\langle\Sigma\rangle$.  There are few particles at small radii in the strongly tapered Zang model so the central values are not reliable.